In [7]:
import pandas as pd
import scanpy as sc
import numpy as np
import os
import anndata 

print(f"Scanpy version : {sc.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

# CosMx donne ces fichiers bruts :
# - exprMat_file.csv     (cellules × gènes)
# - metadata_file.csv    (coordonnées + CellComp)
# - tx_file.csv          (transcripts individuels — optionnel)

Scanpy version : 1.11.5
Pandas version: 2.1.4
Numpy version: 1.26.4


/scratch/local/915645/ipykernel_483010/1857946637.py:7: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"Scanpy version : {sc.__version__}")


In [12]:
# ─── CHOOSE YOUR SAMPLE ──────────────────────────────────────────────────────
SAMPLE = 'half'  # 'half' or 'quarter'

if SAMPLE == 'half':
    BASE_DIR   = "/storage/research/dbmr_luisierlab/projects/cosmx/data/Mouse_Brain/Half_Brain_simple_files"
    OUT_DIR    = "/storage/research/dbmr_luisierlab/projects/cosmx/raw_data_half"
    EXPR_FILE  = os.path.join(BASE_DIR, "Run1000_S1_Half_exprMat_file.csv")
    META_FILE  = os.path.join(BASE_DIR, "Run1000_S1_Half_metadata_file.csv")
    PIXEL_UM   = 0.12028  # µm per pixel — see ExptConfig.txt

elif SAMPLE == 'quarter':
    BASE_DIR   = "/storage/research/dbmr_luisierlab/projects/cosmx/data/Mouse_Brain/Quarter_Brain"
    OUT_DIR    = "/storage/research/dbmr_luisierlab/projects/cosmx/raw_data_quarter"
    EXPR_FILE  = os.path.join(BASE_DIR, "Run5642_S3_Quarter_exprMat_file.csv")
    META_FILE  = os.path.join(BASE_DIR, "Run5642_S3_Quarter_metadata_file.csv")
    PIXEL_UM   = 0.12

else:
    raise ValueError("SAMPLE must be 'half' or 'quarter'")

os.makedirs(OUT_DIR, exist_ok=True)
sc.settings.figdir = OUT_DIR

print(f"Sample   : {SAMPLE}")
print(f"EXPR     : {EXPR_FILE}")
print(f"META     : {META_FILE}")
print(f"OUT_DIR  : {OUT_DIR}")

counts = pd.read_csv(EXPR_FILE)
meta   = pd.read_csv(META_FILE)

# Exclure cell_ID == 0 (transcripts non assignés à une cellule)
counts = counts[counts["cell_ID"] != 0].copy()
meta   = meta[meta["cell_ID"] != 0].copy()

# Construire le cell_uid
counts["cell_uid"] = counts["fov"].astype(int).astype(str) + "_" + counts["cell_ID"].astype(int).astype(str)
meta["cell_uid"]   = meta["fov"].astype(int).astype(str)   + "_" + meta["cell_ID"].astype(int).astype(str)

counts = counts.set_index("cell_uid")
meta   = meta.set_index("cell_uid")

# Séparer gènes / NegProbes / metadata
meta_cols = ["fov", "cell_ID"]
neg_cols  = [c for c in counts.columns if c.startswith("NegPrb")]
gene_cols = [c for c in counts.columns if c not in meta_cols and c not in neg_cols]

# AnnData
adata = sc.AnnData(X=counts[gene_cols].values.astype(np.float32))
adata.obs_names = counts.index.tolist()
adata.var_names = gene_cols

# Aligner meta sur l'ordre de adata
meta = meta.loc[adata.obs_names]

# Coordonnées spatiales en µm
adata.obsm["spatial"] = np.column_stack([
    (meta["CenterX_global_px"].values - meta["CenterX_global_px"].min()) * PIXEL_UM,
    (meta["CenterY_global_px"].values - meta["CenterY_global_px"].min()) * PIXEL_UM,
])

# CellComp si disponible
if "CellComp" in meta.columns:
    adata.obs["CellComp"] = meta["CellComp"].values
else:
    print("Warning: CellComp not found — check metadata columns:")
    print(meta.columns.tolist())

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
adata.write(os.path.join(OUT_DIR, "cosmx_mouse_brain_half.h5ad"))
print(f"Saved: {adata.shape[0]} cells × {adata.shape[1]} genes")

Sample   : half
EXPR     : /storage/research/dbmr_luisierlab/projects/cosmx/data/Mouse_Brain/Half_Brain_simple_files/Run1000_S1_Half_exprMat_file.csv
META     : /storage/research/dbmr_luisierlab/projects/cosmx/data/Mouse_Brain/Half_Brain_simple_files/Run1000_S1_Half_metadata_file.csv
OUT_DIR  : /storage/research/dbmr_luisierlab/projects/cosmx/raw_data_half
['fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.Histone', 'Max.Histone', 'Mean.G', 'Max.G', 'Mean.rRNA', 'Max.rRNA', 'Mean.GFAP', 'Max.GFAP', 'Mean.DAPI', 'Max.DAPI']


/storage/homefs/mj20r066/.local/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


Saved: 48556 cells × 950 genes


In [11]:
print("counts index sample:", counts.index[:5].tolist())
print("meta index sample:  ", meta.index[:5].tolist())

counts index sample: ['1_0', '1_1', '1_2', '1_3', '1_4']
meta index sample:   ['1_1', '1_2', '1_3', '1_4', '1_5']


In [14]:
adata

AnnData object with n_obs × n_vars = 48556 × 950
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'
    obsm: 'spatial'

In [13]:
### 2. Désactiver le module image dans MISO (30 min)
# Dans le code MISO, au lieu de :
model = Miso([rna, protein, image_emb], ...)

# Vous faites simplement :
model = Miso([rna], ...)  
# ou si vous avez RNA + subcellular features :
model = Miso([rna, cell_zone_features], ...)

NameError: name 'Miso' is not defined

In [ ]:
### 3. Sous-sampler pour les tests (important !)
# 90k cellules = trop lent pour débugger
# Commencez avec un petit patch de tissu

# Prendre une région spatiale de 5000 cellules
x_min, x_max = 1000, 3000  # en µm
mask = (
    (meta["x_centroid"] > x_min) & 
    (meta["x_centroid"] < x_max)
)
adata_small = adata[mask][:5000]